In [3]:
#!/usr/bin/env python3
"""
Compare S3 object sets under two prefixes by their relative keys.
Requires: pip install boto3
AWS credentials must be configured (env vars, profile, or instance role).
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Iterable, Set

import boto3
from botocore.config import Config

BUCKET = "seanome-kmerseek"
PREFIX_A = "scope-benchmark/pipeline-outputs/hp_k10-60/"
PREFIX_B = "scope-benchmark/pipeline-outputs/hp/"

LISTING_SAMPLE_TO_PRINT = 20  # how many to print in the console for each diff


def _ensure_trailing_slash(p: str) -> str:
    return p if p.endswith("/") else p + "/"


def iter_keys(bucket: str, prefix: str, s3_client=None) -> Iterable[str]:
    """
    Yield S3 object keys under `prefix` (recursively), skipping "folder" placeholders.
    Uses a paginator, so it works for >1000 keys.
    """
    if s3_client is None:
        s3_client = boto3.client(
            "s3", config=Config(retries={"max_attempts": 10, "mode": "adaptive"})
        )
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            # Skip directory placeholders
            if key.endswith("/") or key == prefix:
                continue
            yield key


def collect_relative_keys(bucket: str, prefix: str, s3_client=None) -> Set[str]:
    """Return the set of keys with the given prefix stripped off."""
    prefix = _ensure_trailing_slash(prefix)
    return {key[len(prefix) :] for key in iter_keys(bucket, prefix, s3_client)}


def main():
    s3 = boto3.client(
        "s3", config=Config(retries={"max_attempts": 10, "mode": "adaptive"})
    )

    pA = _ensure_trailing_slash(PREFIX_A)
    pB = _ensure_trailing_slash(PREFIX_B)

    relA = collect_relative_keys(BUCKET, pA, s3_client=s3)
    relB = collect_relative_keys(BUCKET, pB, s3_client=s3)

    only_in_A = sorted(relA - relB)
    only_in_B = sorted(relB - relA)
    in_both = len(relA & relB)

    print(f"Compared by relative key (suffix after each prefix):")
    print(f"  Bucket: s3://{BUCKET}")
    print(f"  A: {pA}")
    print(f"  B: {pB}\n")

    print(f"Total in A: {len(relA)}")
    print(f"Total in B: {len(relB)}")
    print(f"In both:    {in_both}\n")

    print(f"Only in A ({len(only_in_A)}):")
    for x in only_in_A[:LISTING_SAMPLE_TO_PRINT]:
        print("  ", x)
    if len(only_in_A) > LISTING_SAMPLE_TO_PRINT:
        print("  ...")

    print(f"\nOnly in B ({len(only_in_B)}):")
    for x in only_in_B[:LISTING_SAMPLE_TO_PRINT]:
        print("  ", x)
    if len(only_in_B) > LISTING_SAMPLE_TO_PRINT:
        print("  ...")

    # Write full diffs to files for easy inspection
    with open("only_in_hp_k10-60.txt", "w") as f:
        f.write("\n".join(only_in_A))
    with open("only_in_hp.txt", "w") as f:
        f.write("\n".join(only_in_B))

    print("\nWrote:")
    print("  only_in_hp_k10-60.txt")
    print("  only_in_hp.txt")


# Optional: utility to download a set of relative keys from a given prefix
def download_relative_keys(
    bucket: str,
    prefix: str,
    rel_keys: Iterable[str],
    out_dir: str | os.PathLike,
    s3_client=None,
):
    """
    Download each relative key under `prefix` into `out_dir`, preserving subdirectories.
    Example: download_relative_keys(BUCKET, PREFIX_A, only_in_A, 'out/hp_k10-60')
    """
    if s3_client is None:
        s3_client = boto3.client("s3")
    prefix = _ensure_trailing_slash(prefix)
    out_dir = Path(out_dir)
    for rel in rel_keys:
        src_key = prefix + rel
        dst = out_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        s3_client.download_file(bucket, src_key, str(dst))


if __name__ == "__main__":
    main()

Compared by relative key (suffix after each prefix):
  Bucket: s3://seanome-kmerseek
  A: scope-benchmark/pipeline-outputs/hp_k10-60/
  B: scope-benchmark/pipeline-outputs/hp/

Total in A: 178
Total in B: 189
In both:    162

Only in A (16):
   pipeline_info/execution_report_2024-10-09_15-08-34.html
   pipeline_info/execution_report_2024-10-09_18-51-08.html
   pipeline_info/execution_report_2024-10-11_18-12-00.html
   pipeline_info/execution_report_2024-10-11_22-23-57.html
   pipeline_info/execution_trace_2024-10-09_15-08-34.txt
   pipeline_info/execution_trace_2024-10-09_18-51-08.txt
   pipeline_info/execution_trace_2024-10-11_18-12-00.txt
   pipeline_info/execution_trace_2024-10-11_22-23-57.txt
   pipeline_info/params_2024-10-09_15-08-54.json
   pipeline_info/params_2024-10-09_18-51-27.json
   pipeline_info/params_2024-10-11_18-12-11.json
   pipeline_info/params_2024-10-11_22-24-08.json
   pipeline_info/pipeline_dag_2024-10-09_15-08-34.html
   pipeline_info/pipeline_dag_2024-10-09_18

## So `s3://seanome-kmerseek/scope-benchmark/pipeline-outputs/hp/` has ksizes 5-9 additionally.

## Summary

A one-off S3 bucket diff, not an analysis. It compares two SCOPe benchmark pipeline output prefixes
under `s3://$KMERSEEK_BUCKET/scope-benchmark/pipeline-outputs/`: `hp_k10-60/` (178 objects) against
`hp/` (189 objects), with 162 in common.

The finding is in the markdown note: the `hp/` prefix additionally carries k-sizes 5 through 9. The
16 objects unique to `hp_k10-60/` and 12 of the 27 unique to `hp/` are Nextflow `pipeline_info`
artifacts (execution reports, traces, params, DAGs) from different run dates, not results; the real
difference is the extra multisearch CSVs for k=5-9.

Nothing here needs rerunning. Kept for provenance of which prefix holds which k-range.